# 00 · Analyse exploratoire (EDA) : PlantVillage

Objectifs :
1. inventaire des classes et du déséquilibre ;
2. qualité des données : tailles, doublons, images corrompues ;
3. caractéristiques visuelles (luminosité, couleur, netteté) : ce sont aussi les descripteurs surveillés en production pour la **détection de dérive** ;
4. choix qui en découlent pour la modélisation.

Prérequis : `dvc pull` (ou `python -m src.data.download`) puis `dvc repro prepare`. Lancer le notebook depuis la racine du dépôt.

In [ ]:
import json, sys, random
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image

ROOT = Path.cwd() if (Path.cwd() / "params.yaml").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.utils.common import load_params, EXTENSIONS
from src.monitoring.features import image_stats

params = load_params(ROOT / "params.yaml")
RAW = ROOT / params["data"]["raw_dir"]
PROC = ROOT / params["data"]["processed_dir"]
print("images :", RAW)

## 1. Classes et déséquilibre

In [ ]:
counts = pd.Series({d.name: sum(1 for p in d.iterdir() if p.suffix.lower() in EXTENSIONS)
                    for d in sorted(RAW.iterdir()) if d.is_dir()}).sort_values()
plants = counts.groupby(counts.index.str.split("___").str[0]).sum().sort_values()
healthy = counts[counts.index.str.contains("healthy")].sum()
print(f"{len(counts)} classes, {counts.sum()} images, {len(plants)} espèces")
print(f"min {counts.min()} ({counts.idxmin()}) / max {counts.max()} ({counts.idxmax()}) -> rapport {counts.max() / counts.min():.0f}:1")
print(f"plantes saines : {healthy} ({100 * healthy / counts.sum():.1f} %)")
fig, ax = plt.subplots(1, 2, figsize=(15, 9))
counts.plot.barh(ax=ax[0], color="#3E5A27"); ax[0].set_title("Images par classe")
plants.plot.barh(ax=ax[1], color="#8DB33A"); ax[1].set_title("Images par espèce")
plt.tight_layout(); plt.show()

**Conclusion** : fort déséquilibre, donc métrique principale **F1 macro**, **poids de classes** ou **Focal Loss**, et split **stratifié**.

## 2. Qualité : doublons, fuites, tailles

In [ ]:
summary_path = PROC / "summary.json"
if summary_path.exists():
    s = json.loads(summary_path.read_text())
    for k in ("readable_images", "corrupt_images_removed", "contradictory_labels_removed", "exact_duplicates_removed",
              "near_duplicate_groups", "kept_images", "train", "val", "test"):
        print(f"{k:32s} {s.get(k)}")
else:
    print("lancez `dvc repro prepare` pour obtenir le résumé de nettoyage")

In [ ]:
random.seed(0)
all_paths = [p for d in RAW.iterdir() if d.is_dir() for p in d.iterdir() if p.suffix.lower() in EXTENSIONS]
sample = random.sample(all_paths, min(600, len(all_paths)))
sizes = pd.Series([Image.open(p).size for p in sample]).value_counts()
print("tailles d'image les plus fréquentes :"); print(sizes.head())

Les images font presque toutes 256×256 : un redimensionnement en 224×224 ne déforme pas les feuilles.

## 3. Caractéristiques visuelles (descripteurs de dérive)

In [ ]:
rows = []
for p in sample:
    st = image_stats(Image.open(p))
    st["label"] = p.parent.name
    st["healthy"] = "healthy" in p.parent.name
    rows.append(st)
feat = pd.DataFrame(rows)
display(feat.describe().T[["mean", "std", "min", "max"]].round(3))
cols = ["brightness", "contrast", "saturation", "green_ratio", "sharpness", "leaf_coverage"]
fig, axes = plt.subplots(2, 3, figsize=(15, 7))
for ax, c in zip(axes.ravel(), cols):
    for h, color in ((True, "#2A9D8F"), (False, "#E4572E")):
        ax.hist(feat.loc[feat.healthy == h, c], bins=30, alpha=0.6, color=color, label="saine" if h else "malade")
    ax.set_title(c); ax.legend()
plt.tight_layout(); plt.show()

**Lecture** :
- les feuilles malades ont un `green_ratio` plus faible (taches jaunes ou brunes), ce qui valide l'heuristique de sévérité ;
- luminosité, contraste et fond sont très homogènes : ce sont des **photos de laboratoire**. Des photos prises au champ s'écarteront de ces distributions. C'est exactement ce que le flow `monitoring-check` (Evidently) compare en production avec `reports/monitoring/reference.csv`.

## 4. Aperçu

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(16, 8))
for ax, p in zip(axes.ravel(), random.sample(all_paths, 18)):
    ax.imshow(Image.open(p)); ax.set_title(p.parent.name.replace("___", "\n"), fontsize=7); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Décisions pour la modélisation
| Constat | Décision |
|---|---|
| Déséquilibre jusqu'à environ 35:1 | F1 macro, poids de classes ou Focal Loss, split stratifié |
| Doublons et quasi-doublons | déduplication MD5 et groupes dHash **avant** le split (anti-fuite) |
| Fond uni, lumière constante | augmentations (couleur, RandAugment, CutMix), surveillance de la dérive en production |
| Images de 256 px | entrée en 224 px, réseaux pré-entraînés sur ImageNet (transfer learning) |